In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%load_ext tensorboard

In [ ]:
import sys
import logging

In [ ]:
import mlflow

In [ ]:
import main

In [ ]:
import torch

In [ ]:
_handler = logging.StreamHandler(sys.stdout)
_handler.setLevel(logging.DEBUG)
_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s", datefmt="%H:%M:%S'"
    )
)
_logger = logging.getLogger("adaptive_milling_training")
for _ in _logger.handlers:
    _logger.removeHandler(_)
_logger.setLevel(logging.DEBUG)
_logger.addHandler(_handler)


In [ ]:
from utils import MONAI_LOG_DIR

mlflow.pytorch.autolog()
port = 54598
mlflow_uri = f"file://{MONAI_LOG_DIR}"
_logger.info("Setting up mlflow with URI '%s'", mlflow_uri)
mlflow.set_tracking_uri(mlflow_uri)
mlflow.set_experiment("MONAI adaptive milling")

print(
    f"Run the following command to start:\n$mlflow ui --backend-store-uri {mlflow_uri} --port {port}\nThen navigate to:\nhttp://127.0.0.1:{port}"
)


In [ ]:
%%script false --no-raise-error
from utils import TENSORBOARD_LOG_DIR

logging.info("Setting up Tensorboard with log dir %s", TENSORBOARD_LOG_DIR)
%tensorboard --logdir $TENSORBOARD_LOG_DIR

In [ ]:
test_csv = "/ceph/groups/structbio/adaptive_milling_project/2024labels_new/test.csv"
all_files_csv = (
    "/ceph/groups/structbio/adaptive_milling_project/2024labels_new/all_files.csv"
)


In [ ]:
try:
    main.run_training(
        all_files_csv,
        model_name="segresnet",
        loss_name="diceloss",
        learning_rate=6e-3,
        epochs=30,
    )
except:  # noqa: E722
    with torch.no_grad():
        torch.cuda.empty_cache()
    try:
        mlflow.end_run()
    except:  # noqa: E722
        _logger.error("Failed to end MLFlow run", exc_info=True)
    raise